# 10. 이상치는 크기가 아니라 어긋남으로 가른다

> 2026-09-07 · 이동원 · 이슈 [#132](https://github.com/devlee328288/Alpha_Stack/issues/132) ·
> 결론 문서: [데이터파트 v3.9 변경사항](../../docs/데이터파트/version3.9/변경사항.md) (예정)

오준영 님이 개별종목 후보군(업종 10 × 종목 5)에 `|adj_close 일간수익률| > 100%` 필터를 걸었더니
**1건**이 지워졌습니다. 이 노트북은 그 1건을 **먼저 그대로 재현**하고, 그것이 데이터 오류인지
진짜 사건인지를 KRX 등락률로 가른 뒤, 판별자를 "크기" 에서 "**거래소 등락률과의 어긋남**" 으로
바꿨을 때 무엇이 달라지는지를 잽니다.

재다 보니 어긋난 행의 원인이 **재개일 조정 규칙**에 있었습니다 — 인적분할·회생 재상장일에
"주식수 배율 = 가격 배율" 을 적용하고 있었습니다. 우리 값을 실제로 정하는 chain 구간 사건을
DART 공시로 전수 대조하고, 규칙을 고친 전후를 같은 자리에서 잽니다.

순서:

1. 재현 — 필터가 지운 1건은 무엇인가
2. 허용폭 — 어긋남을 어디서 자르나 (0.15%p 인가 1%p 인가)
3. 판별 함수 — `supply.adj_quality` 를 후보군에 붙이면
4. 재개일 규칙 — 계수가 바뀌는 사건은 무슨 사건이었나 (DART)
5. 전후 — 규칙을 고치면 그 사건들이 KRX 와 맞는가

In [1]:
import sqlite3
import sys
import time
from fractions import Fraction
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "supply").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from common import corporate_actions as ca                          # noqa: E402, I001
from supply.adj_quality import attach_adjustment_quality            # noqa: E402
from supply.stock_training_universe import (                         # noqa: E402
    build_sector_candidate_frame, filter_extreme_adjusted_returns)

DB = ROOT / "data" / "krx_cache.db"
# HF 배포본을 내려받아 둔 자리. 없으면 가장 최근 반출 폴더.
OUTBOX = ROOT / "data" / "outbox" / "hf_snapshot" / "full"
if not OUTBOX.exists():
    OUTBOX = sorted((ROOT / "data" / "outbox").glob("2*/full/daily_price_dev.parquet"))[-1].parent
print("DB:", DB, "· 반출본:", OUTBOX)
con = sqlite3.connect(DB)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 80)
FLAGS = ["bas_dd", "code", "name", "adj_return_1d", "adj_change_rate_gap",
         "is_adj_suspect", "is_extreme_return"]

DB: C:\Users\kik32\workspace\EST-Camp-AI-Quant\team_project\Alpha_Stack\data\krx_cache.db · 반출본: C:\Users\kik32\workspace\EST-Camp-AI-Quant\team_project\Alpha_Stack\data\outbox\hf_snapshot\full


## 1. 재현 — 필터가 지운 1건

오준영 님 경로 그대로입니다: 반출본(개발구간) → 후보군 → `filter_extreme_adjusted_returns`.

In [2]:
t0 = time.time()
daily = pd.read_parquet(OUTBOX / "daily_price_dev.parquet")
index = pd.read_parquet(OUTBOX / "index_price_dev.parquet")
cand = build_sector_candidate_frame(daily, index)
filt = filter_extreme_adjusted_returns(cand, daily)
print(f"후보군 {len(cand):,}행 · 필터 뒤 {len(filt):,}행 · {time.time() - t0:.0f}초")
print(filt.attrs["extreme_return_filter"])

removed = cand.merge(filt[["bas_dd", "code"]], how="left", indicator=True)
removed = removed[removed["_merge"] == "left_only"]
removed[["bas_dd", "code", "name", "industry"]]

후보군 176,705행 · 필터 뒤 176,704행 · 23초
{'absolute_limit': 1.0, 'source_extreme_rows': 76, 'removed_candidate_rows': 1}


,bas_dd,code,name,industry
75916,20160511,008020,경남에너지,전기·가스


지워진 행 앞뒤를 열어 봅니다. **우리 수정주가 수익률과 KRX 등락률이 같으면** 그날 실제로 그렇게
움직인 것이고, 다르면 우리 조정이 틀린 것입니다.

In [3]:
def 이웃(code, bas_dd, 앞=2, 뒤=3):
    rows = pd.read_sql_query(
        "SELECT bas_dd, close, adj_close, adj_source, change_rate, volume, listed_shares "
        "FROM daily_price WHERE code=? ORDER BY bas_dd", con, params=(code,))
    i = rows.index[rows.bas_dd == bas_dd][0]
    out = rows.iloc[max(0, i - 앞): i + 뒤 + 1].copy()
    out["adj수익률"] = (out.adj_close / out.adj_close.shift(1) - 1) * 100
    out["gap"] = (out["adj수익률"] - out.change_rate).abs()
    return out

이웃("008020", "20160511")

,bas_dd,close,adj_close,adj_source,change_rate,volume,listed_shares,adj수익률,gap
1570,20160509,10150,10150.0,fdr,0.00,0,41249008,NaN,NaN
1571,20160510,10250,10250.0,fdr,0.99,453925,41249008,0.985222,0.004778
1572,20160511,26000,26000.0,fdr,153.66,593366,41249008,153.658537,0.001463
1573,20160512,22800,22800.0,fdr,-12.31,297773,41249008,-12.307692,0.002308
1574,20160513,17200,17200.0,fdr,-24.56,366263,41249008,-24.561404,0.001404
1575,20160516,13900,13900.0,fdr,-19.19,267174,41249008,-19.186047,0.003953


+153.66% 와 KRX +153.66% — **일치합니다.** 경남에너지는 데이터 오류가 아니라 진짜 사건이고,
크기 필터는 이 행을 지웁니다. 반면 크기 필터가 **못 잡는** 오류가 있습니다.

In [4]:
이웃("004170", "20110610")     # 신세계 · 인적분할 재상장일 — 재생성 전엔 −60.6% 였다

,bas_dd,close,adj_close,adj_source,change_rate,volume,listed_shares,adj수익률,gap
356,20110608,270000,354500.0,chain,0.00,0,37721000,NaN,NaN
357,20110609,270000,354500.0,chain,0.00,0,37721000,0.000000,0.000000
358,20110610,407500,407500.0,chain,14.95,322652,9845181,14.950635,0.000635
359,20110613,366500,366500.0,chain,-10.06,174250,9845181,-10.061350,0.001350
360,20110614,349000,349000.0,chain,-4.77,182827,9845181,-4.774898,0.004898
361,20110615,340000,340000.0,chain,-2.58,144077,9845181,-2.578797,0.001203


## 2. 허용폭 — 어긋남을 어디서 자르나

`build_adj_prices.verify()` ① 은 0.15%p 로 **비율**을 봅니다. 행 플래그에도 같은 값을 쓰면
어떻게 되는지, 개발구간 전 시장에서 잽니다 (SQL · 1~2분).

In [5]:
t0 = time.time()
gap = pd.read_sql_query('''
WITH seq AS (
  SELECT code, bas_dd, close, adj_close, change_rate, adj_source,
         LAG(adj_close) OVER (PARTITION BY code ORDER BY bas_dd) AS prev_adj
  FROM daily_price WHERE adj_close IS NOT NULL AND close > 0 AND bas_dd < '20240901'
)
SELECT code, bas_dd, close, adj_source, change_rate,
       (adj_close / prev_adj - 1) * 100 AS adj_ret,
       ABS((adj_close / prev_adj - 1) * 100 - change_rate) AS gap
FROM seq WHERE prev_adj IS NOT NULL AND prev_adj > 0 AND change_rate IS NOT NULL AND
      ABS((adj_close / prev_adj - 1) * 100 - change_rate) > 0.15
''', con)
n = con.execute(
    "SELECT COUNT(*) FROM daily_price "
    "WHERE adj_close IS NOT NULL AND bas_dd < '20240901'").fetchone()[0]
print(f"비교 가능 행 약 {n:,} · gap > 0.15 인 행 {len(gap):,} · {time.time() - t0:.0f}초")
표 = pd.DataFrame({"허용폭(%p)": [0.15, 0.3, 0.5, 1.0, 2.0, 5.0, 30.0]})
표["어긋남 행"] = [int((gap.gap > t).sum()) for t in 표["허용폭(%p)"]]
표["비율(%)"] = (표["어긋남 행"] / n * 100).round(3)
표

비교 가능 행 약 7,888,945 · gap > 0.15 인 행 13,622 · 86초


,허용폭(%p),어긋남 행,비율(%)
0,0.15,13622,0.173
1,0.30,6008,0.076
2,0.50,3390,0.043
3,1.00,1353,0.017
4,2.00,699,0.009
5,5.00,293,0.004
6,30.00,96,0.001


0.15 와 1.0 사이 띠가 어떤 행인지 — 가격대를 봅니다. 저가주에 몰려 있으면 반올림입니다
(FDR 수정가격은 원 단위, 500원 종목에서 1원은 0.2%).

In [6]:
띠 = gap[(gap.gap > 0.15) & (gap.gap <= 1.0)]
bins = [0, 500, 1000, 5000, 10000, 100000, 10**9]
분포 = pd.cut(띠.close, bins).value_counts().sort_index()
저가 = int(분포.iloc[:3].sum())
print(f"0.15 < gap ≤ 1.0: {len(띠):,}행 · 5,000원 미만 {저가:,} ({저가 / len(띠) * 100:.0f}%)")
분포

0.15 < gap ≤ 1.0: 12,269행 · 5,000원 미만 11,559 (94%)


close
(0, 500]                2945
(500, 1000]             2985
(1000, 5000]            5629
(5000, 10000]            629
(10000, 100000]           80
(100000, 1000000000]       1
Name: count, dtype: int64

→ 행 플래그의 허용폭은 **1%p**. `verify()` 의 0.15 는 집계 게이트라 그대로 둡니다. 두 상수는
각자 자리에서 근거를 적었습니다 (`supply/adj_quality.py` 모듈 설명).

## 3. 판별 함수 — 후보군에 붙이면

`supply.adj_quality.attach_adjustment_quality(candidates, daily)` 는 오준영 님 필터와 같은 서명이고,
칸 넷(`adj_return_1d` · `adj_change_rate_gap` · `is_adj_suspect` · `is_extreme_return`)을 붙입니다.
먼저 **반출본(HF 배포본 = 재생성 전 값)** 으로.

In [7]:
t0 = time.time()
붙임_전 = attach_adjustment_quality(cand, daily)
print(f"{time.time() - t0:.0f}초 ·", 붙임_전.attrs["adjustment_quality"])
붙임_전.loc[붙임_전.is_adj_suspect | 붙임_전.is_extreme_return, FLAGS]

13초 · {'gap_tolerance': 1.0, 'extreme_pct': 30.0, 'rows': 7888945, 'comparable_rows': 7885483, 'suspect_rows': 1406, 'extreme_rows': 2801, 'candidate_rows': 176705, 'candidate_suspect_rows': 2, 'candidate_extreme_rows': 5, 'candidate_unmatched_rows': 0}


,bas_dd,code,name,adj_return_1d,adj_change_rate_gap,is_adj_suspect,is_extreme_return
17268,20110610,004170,신세계,-60.608333,7.555833e+01,True,False
44070,20130829,035420,NAVER,63.543441,5.919344e+01,True,False
75916,20160511,008020,경남에너지,153.658537,1.463415e-03,False,True
76431,20160525,003520,영진약품,-30.000000,3.552714e-15,False,True
122462,20200331,003000,부광약품,30.005526,5.525880e-03,False,True
126135,20200720,019170,신풍제약,30.000000,3.552714e-15,False,True
126380,20200727,019170,신풍제약,-30.000000,3.552714e-15,False,True


반출본이 재생성 **전** 값이면 의심 2행(신세계 · NAVER)이 보이고, 재생성 뒤 반출본이면 0행입니다.
어느 쪽이든 경남에너지를 포함한 진짜 극단 사건은 `is_extreme_return` 으로 **남습니다.**
같은 후보군을 **DB(재생성 후)** 값으로 다시 붙여 봅니다.

In [8]:
t0 = time.time()
daily_db = pd.read_sql_query(
    "SELECT bas_dd, code, close, adj_close, change_rate FROM daily_price "
    "WHERE bas_dd < '20240901'", con)
붙임_후 = attach_adjustment_quality(cand, daily_db)
print(f"{time.time() - t0:.0f}초 ·", 붙임_후.attrs["adjustment_quality"])
붙임_후.loc[붙임_후.is_adj_suspect | 붙임_후.is_extreme_return, FLAGS]

33초 · {'gap_tolerance': 1.0, 'extreme_pct': 30.0, 'rows': 7888945, 'comparable_rows': 7885483, 'suspect_rows': 1353, 'extreme_rows': 2801, 'candidate_rows': 176705, 'candidate_suspect_rows': 0, 'candidate_extreme_rows': 5, 'candidate_unmatched_rows': 0}


,bas_dd,code,name,adj_return_1d,adj_change_rate_gap,is_adj_suspect,is_extreme_return
75916,20160511,008020,경남에너지,153.658537,1.463415e-03,False,True
76431,20160525,003520,영진약품,-30.000000,3.552714e-15,False,True
122462,20200331,003000,부광약품,30.005526,5.525880e-03,False,True
126135,20200720,019170,신풍제약,30.000000,3.552714e-15,False,True
126380,20200727,019170,신풍제약,-30.000000,3.552714e-15,False,True


## 4. 재개일 규칙 — 계수가 바뀌는 사건은 무슨 사건이었나

의심 행이 **둘 다 chain 구간의 재개일**이었습니다. 재개일에 상장주식수가 바뀐 사건을 전 시장에서
모아(SQL · 약 2분) **옛 규칙과 새 규칙이 다른 계수를 내는 사건**을 코드 자체로 찾습니다.

- 옛 규칙: 주식수가 1.5배 넘게 움직였으면 그 배율의 역수, 아니면 1
- 새 규칙: `corporate_actions.adjustment_factor` — 기준가가 배율의 50~150% 안이면 배율(전과 같음),
  밖이면 **기준가/전일종가**, 1.1~1.5배 변동도 기준가
- `k` = 앞주식수/주식수 (감자면 >1) · `b` = KRX 기준가/전일종가

In [9]:
t0 = time.time()
ev = pd.read_sql_query('''
WITH seq AS (
  SELECT code, name, bas_dd, close, change, change_rate, volume, open, listed_shares,
         adj_source,
         LAG(close) OVER w AS p_close, LAG(volume) OVER w AS p_vol,
         LAG(open) OVER w AS p_open, LAG(listed_shares) OVER w AS p_sh
  FROM daily_price WINDOW w AS (PARTITION BY code ORDER BY bas_dd)
)
SELECT * FROM seq
WHERE p_vol = 0 AND p_open = 0 AND volume > 0 AND p_close > 0 AND close > 0
  AND listed_shares IS NOT NULL AND p_sh IS NOT NULL AND listed_shares != p_sh
  AND change IS NOT NULL
''', con)
ev["k"] = ev.p_sh / ev.listed_shares
ev["b"] = (ev.close - ev.change) / ev.p_close


def 옛계수(r):
    배율 = Fraction(int(r.listed_shares), int(r.p_sh))
    큰변동 = 배율 >= Fraction(3, 2) or 배율 <= Fraction(2, 3)
    return 1 / 배율 if 큰변동 else Fraction(1)


def 새계수(r):
    앞 = {"open": 0, "high": 0, "low": 0, "close": r.p_close, "listed_shares": r.p_sh}
    뒤 = {"close": r.close, "change": r.change, "listed_shares": r.listed_shares}
    return ca.adjustment_factor(앞, 뒤)


ev["옛계수"] = [옛계수(r) for r in ev.itertuples()]
ev["새계수"] = [새계수(r) for r in ev.itertuples()]
ev["바뀜"] = ev["옛계수"] != ev["새계수"]
print(f"재개일 + 주식수 변동 {len(ev):,}건 · 계수가 바뀌는 사건 {int(ev['바뀜'].sum())}건"
      f" · {time.time() - t0:.0f}초")
바뀜 = ev[ev["바뀜"]]
원천 = 바뀜.adj_source.str.replace(r"\+ca_fix$", "", regex=True)
print("바뀌는 사건의 adj_source:", 원천.value_counts().to_dict())

재개일 + 주식수 변동 1,305건 · 계수가 바뀌는 사건 223건 · 106초
바뀌는 사건의 adj_source: {'fdr': 171, 'chain': 52}


FDR 구간(fdr)은 FDR 이 스스로 기준가만큼 펴 놓아 우리 규칙이 값을 정하지 않습니다(5절에서
확인). 실제로 우리 값을 정하는 것은 **chain 구간**뿐이라, 그 사건들을 DART 공시 제목으로
확인합니다. DART 수집은 2010-01-04 부터라 2010년 초 사건은 공시가 비어 있을 수 있습니다.

In [10]:
chain = 바뀜[바뀜.adj_source.str.startswith("chain")].sort_values("bas_dd").copy()
패턴 = ["분할", "감자", "합병", "회생", "출자전환", "재상장"]


def 공시(code, bas_dd):
    lo = (pd.Timestamp(bas_dd) - pd.Timedelta(days=400)).strftime("%Y%m%d")
    rows = con.execute(
        "SELECT rcept_dt, report_nm FROM dart_disclosure "
        "WHERE stock_code=? AND rcept_dt BETWEEN ? AND ? ORDER BY rcept_dt",
        (code, lo, bas_dd)).fetchall()
    hits = [n for _, n in rows if any(p in n for p in 패턴)]
    return " · ".join(dict.fromkeys(n[:28] for n in hits[-3:])) if hits else "(공시 없음)"


def 유형(공시문, b):
    if "분할" in 공시문:
        return "① 인적분할·지주전환"
    if ("(공시 없음)" in 공시문 or "합병등" in 공시문) and b < 1:
        return "① 인적분할·지주전환 (추정 · 가치가 줄었다)"
    return "② 회생·합병+감자"


chain["관련 공시"] = [공시(c, d) for c, d in zip(chain.code, chain.bas_dd, strict=True)]
chain["유형"] = [유형(s, b) for s, b in zip(chain["관련 공시"], chain.b, strict=True)]
print(f"chain 구간 {len(chain)}건 · 종목 {chain.code.nunique()}")
print(chain["유형"].value_counts().to_string())
chain[["code", "name", "bas_dd", "k", "b", "change_rate", "유형", "관련 공시"]].round(3)

chain 구간 52건 · 종목 51
유형
② 회생·합병+감자                    28
① 인적분할·지주전환                   21
① 인적분할·지주전환 (추정 · 가치가 줄었다)     3


,code,name,bas_dd,k,b,change_rate,유형,관련 공시
380,012170,한신DNP,20100113,30.000,60.000,-15.00,② 회생·합병+감자,(공시 없음)
31,000590,CS홀딩스,20100127,1.265,0.580,14.96,① 인적분할·지주전환 (추정 · 가치가 줄었다),[기재정정]증권발행실적보고서(합병등)
328,009440,KC그린홀딩스,20100129,1.176,0.920,-15.00,① 인적분할·지주전환,최대주주등소유주식변동신고서(회사분할로 인한 변동) · 증권발행실적보고서(합병등) ·...
99,002020,코오롱,20100201,3.571,0.693,-10.91,① 인적분할·지주전환 (추정 · 가치가 줄었다),증권발행실적보고서(합병등) · [기재정정]증권발행실적보고서(합병등)
100,002025,코오롱우,20100201,3.571,0.679,-1.11,① 인적분할·지주전환 (추정 · 가치가 줄었다),(공시 없음)
168,003620,쌍용차,20100212,3.344,7.600,-10.53,② 회생·합병+감자,[기재정정]주요사항보고서(감자결정)
960,088800,에이스테크,20100325,0.479,0.811,-4.56,① 인적분할·지주전환,증권발행실적보고서(합병등) · 주권매매거래정지해제(인적분할후 존속법인에 대한 감자
34,000680,LS네트웍스,20100511,1.102,1.005,-4.57,② 회생·합병+감자,[기재정정]주요사항보고서(감자결정)
508,025530,SJM홀딩스,20100531,1.934,0.741,-15.00,① 인적분할·지주전환,[기재정정]증권신고서(분할) · 증권발행실적보고서(합병등) · [기재정정]증권발행실...
155,003350,한국화장품제조,20100601,4.545,1.267,15.00,① 인적분할·지주전환,[기재정정]증권신고서(분할) · 증권발행실적보고서(합병등) · 최대주주등소유주식변동...


## 5. 전후 — 규칙을 고치면 그 사건들이 KRX 와 맞는가

옛 계수와 새 계수를 같은 원가격에 적용해 재개일 수익률을 냅니다. **DB 는 이미 새 규칙으로
재생성된 값**이라 `adj수익률_DB` 가 KRX 와 같아야 합니다. FDR 구간 사건도 같은 표로 확인합니다 —
FDR 값이 이미 KRX 와 맞으면 규칙 변경이 그 구간엔 아무 영향이 없다는 뜻입니다.

In [11]:
키 = ["bas_dd", "open", "high", "low", "close", "change", "change_rate",
      "volume", "listed_shares", "adj_close"]
SQL = ("SELECT " + ", ".join(키) + " FROM daily_price WHERE code=? AND bas_dd {} ? "
       "ORDER BY bas_dd DESC LIMIT 1")


def 전후표(사건들):
    rows = []
    for e in 사건들.itertuples():
        앞 = dict(zip(키, con.execute(SQL.format("<"), (e.code, e.bas_dd)).fetchone(), strict=True))
        뒤 = dict(zip(키, con.execute(SQL.format("="), (e.code, e.bas_dd)).fetchone(), strict=True))
        옛, 새 = float(e.옛계수), float(e.새계수)
        원가격비 = 뒤["close"] / 앞["close"]
        rows.append({
            "code": e.code, "name": e.name, "bas_dd": e.bas_dd,
            "유형": getattr(e, "유형", ""),
            "옛계수": round(옛, 3), "새계수": round(새, 3),
            "수익률_옛규칙": round((원가격비 / 옛 - 1) * 100, 2),
            "수익률_새규칙": round((원가격비 / 새 - 1) * 100, 2),
            "adj수익률_DB": round((뒤["adj_close"] / 앞["adj_close"] - 1) * 100, 2),
            "KRX": e.change_rate,
        })
    out = pd.DataFrame(rows)
    out["gap_DB"] = (out["adj수익률_DB"] - out["KRX"]).abs()
    return out


전후 = 전후표(chain)
초과 = int((전후.gap_DB > 1).sum())
print(f"chain {len(전후)}건 — DB 값과 KRX 의 gap 최대 {전후.gap_DB.max():.4f}%p"
      f" · 1%p 초과 {초과}건")
전후

chain 52건 — DB 값과 KRX 의 gap 최대 0.0000%p · 1%p 초과 0건


,code,name,bas_dd,유형,옛계수,새계수,수익률_옛규칙,수익률_새규칙,adj수익률_DB,KRX,gap_DB
0,012170,한신DNP,20100113,② 회생·합병+감자,30.000,60.000,70.00,-15.00,-15.00,-15.00,0.0
1,000590,CS홀딩스,20100127,① 인적분할·지주전환 (추정 · 가치가 줄었다),1.000,0.580,-33.31,14.96,14.96,14.96,0.0
2,009440,KC그린홀딩스,20100129,① 인적분할·지주전환,1.000,0.920,-21.84,-15.00,-15.00,-15.00,0.0
3,002020,코오롱,20100201,① 인적분할·지주전환 (추정 · 가치가 줄었다),3.571,0.693,-82.71,-10.91,-10.91,-10.91,0.0
4,002025,코오롱우,20100201,① 인적분할·지주전환 (추정 · 가치가 줄었다),3.571,0.679,-81.19,-1.11,-1.11,-1.11,0.0
5,003620,쌍용차,20100212,② 회생·합병+감자,3.344,7.600,103.32,-10.53,-10.53,-10.53,0.0
6,088800,에이스테크,20100325,① 인적분할·지주전환,0.479,0.811,61.41,-4.56,-4.56,-4.56,0.0
7,000680,LS네트웍스,20100511,② 회생·합병+감자,1.000,1.005,-4.13,-4.57,-4.57,-4.57,0.0
8,025530,SJM홀딩스,20100531,① 인적분할·지주전환,1.934,0.741,-67.41,-15.00,-15.00,-15.00,0.0
9,003350,한국화장품제조,20100601,① 인적분할·지주전환,4.545,1.267,-67.93,15.00,15.00,15.00,0.0


In [12]:
fdr전후 = 전후표(바뀜[바뀜.adj_source.str.startswith("fdr")])
초과 = int((fdr전후.gap_DB > 1).sum())
print(f"fdr {len(fdr전후)}건 — DB(FDR) 값과 KRX 의 gap 1%p 초과 {초과}건 "
      f"(FDR 이 이미 같은 방식으로 펴 놓았다)")
fdr전후[fdr전후.gap_DB > 1]

fdr 171건 — DB(FDR) 값과 KRX 의 gap 1%p 초과 1건 (FDR 이 이미 같은 방식으로 펴 놓았다)


,code,name,bas_dd,유형,옛계수,새계수,수익률_옛규칙,수익률_새규칙,adj수익률_DB,KRX,gap_DB
10,000480,시알홀딩스,20230728,,1.0,1.033,-27.59,-29.91,-27.59,-29.91,2.32


## 결론

| 물음 | 답 |
|---|---|
| 필터가 지운 1건은? | **진짜 사건** — 남긴다 (경남에너지 +153.66% = KRX +153.66%) |
| 판별자는? | 크기가 아니라 **KRX 등락률과의 어긋남** |
| 허용폭은? | 행 플래그 **1%p** (0.15~1.0 띠는 저가주 반올림) · verify() 는 0.15 그대로 |
| 어긋난 원인은? | 재개일 규칙이 인적분할·회생 재상장에 감자 규칙을 적용 (4절 표) |
| 고치면? | chain 사건 전부 gap 0 · FDR 구간 무영향 · 후보군 의심 0행 |

다음: 이 노트북의 결론을 [데이터파트 v3.9](../../docs/데이터파트/) 에 옮기고, 반출본을 재배포합니다
(재생성으로 값이 바뀌었으므로 `verify_hf_dataset.py` 가 재배포 필요를 냅니다).